# 阶段 1：CSMAR 原始数据审计

只读检查 `项目初始数据/`。本 Notebook 不复制、清洗或改写原始数据，也不实现任何因子或回测逻辑。

In [1]:
from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from datetime import date
from itertools import combinations
from pathlib import Path

ROOT = Path.cwd()
RAW_DIR = ROOT / '项目初始数据'
assert RAW_DIR.is_dir(), f'未找到原始数据目录: {RAW_DIR}'

FIELD_DECISIONS = {
    'strategy_return_field': 'Dretwd',
    'momentum_price_field': 'Adjprcwd',
    'pe_field': 'PE1TTM',
    'benchmark_code': 'H00300',
}

def dataset_kind(path: Path) -> str:
    name = path.name
    if name.startswith('TRD_Dalyr'):
        return 'daily_trading'
    if name.startswith('STK_MKT_ValuationMetrics'):
        return 'valuation'
    if name == 'IDX_Idxtrd.csv':
        return 'index'
    return {
        'TRD_Co.csv': 'stock_master',
        'TRD_Cale.csv': 'calendar',
        'FS_Comins.csv': 'income_statement',
        'FS_Combas.csv': 'balance_sheet',
        'IAR_Forecdt.csv': 'disclosure_date',
    }.get(name, 'other')

KEY_FIELDS = {
    'daily_trading': ('Stkcd', 'Trddt'),
    'valuation': ('Symbol', 'TradingDate'),
    'index': ('Indexcd', 'Idxtrd01'),
    'stock_master': ('Stkcd',),
    'calendar': ('Markettype', 'Clddt'),
    'income_statement': ('Stkcd', 'Accper', 'Typrep'),
    'balance_sheet': ('Stkcd', 'Accper', 'Typrep'),
    'disclosure_date': ('Stkcd', 'Accper'),
}
DATE_FIELD = {
    'daily_trading': 'Trddt', 'valuation': 'TradingDate', 'index': 'Idxtrd01',
    'stock_master': 'Listdt', 'calendar': 'Clddt', 'income_statement': 'Accper',
    'balance_sheet': 'Accper', 'disclosure_date': 'Accper',
}
NUMERIC_FIELDS = {
    'daily_trading': ('Opnprc', 'Clsprc', 'Dretwd', 'Adjprcwd', 'Dnshrtrd', 'Dnvaltrd'),
    'valuation': ('PE1TTM',),
    'index': ('Idxtrd05', 'Idxtrd08'),
    'income_statement': ('B002000000',),
    'balance_sheet': ('A003000000',),
}

def open_csv(path: Path):
    # CSMAR exports in this delivery are UTF-8; fallback preserves a readable failure reason.
    for encoding in ('utf-8-sig', 'gb18030'):
        try:
            handle = path.open('r', encoding=encoding, newline='')
            header = handle.readline()
            handle.seek(0)
            if header:
                return handle, encoding
            handle.close()
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('unknown', b'', 0, 1, '无法使用 UTF-8 或 GB18030 读取')

def parse_date(value: str) -> str | None:
    try:
        return date.fromisoformat(value).isoformat()
    except (TypeError, ValueError):
        return None

def scan_file(path: Path) -> dict:
    kind = dataset_kind(path)
    key_fields = KEY_FIELDS.get(kind, ())
    date_field = DATE_FIELD.get(kind)
    numeric_fields = NUMERIC_FIELDS.get(kind, ())
    record_count = invalid_date_count = duplicate_key_count = 0
    dates, stocks, keys = [], set(), set()
    missing = Counter()
    non_numeric = Counter()
    numeric = {field: {'min': None, 'max': None, 'nonpositive': 0} for field in numeric_fields}
    distributions = {'Markettype': Counter(), 'Trdsta': Counter(), 'Typrep': Counter()}

    handle, encoding = open_csv(path)
    with handle:
        reader = csv.DictReader(handle)
        fields = reader.fieldnames or []
        for row in reader:
            record_count += 1
            for field in fields:
                if not (row.get(field) or '').strip():
                    missing[field] += 1
            if 'Stkcd' in fields and (code := (row.get('Stkcd') or '').strip()):
                stocks.add(code)
            elif 'Symbol' in fields and (code := (row.get('Symbol') or '').strip()):
                stocks.add(code)
            if date_field:
                parsed = parse_date((row.get(date_field) or '').strip())
                if parsed is None:
                    invalid_date_count += 1
                else:
                    dates.append(parsed)
            if key_fields:
                key = tuple((row.get(field) or '').strip() for field in key_fields)
                if key in keys:
                    duplicate_key_count += 1
                else:
                    keys.add(key)
            for field in numeric_fields:
                value = (row.get(field) or '').strip()
                if not value:
                    continue
                try:
                    number = float(value)
                except ValueError:
                    non_numeric[field] += 1
                    continue
                numeric[field]['min'] = number if numeric[field]['min'] is None else min(numeric[field]['min'], number)
                numeric[field]['max'] = number if numeric[field]['max'] is None else max(numeric[field]['max'], number)
                if number <= 0:
                    numeric[field]['nonpositive'] += 1
            for field, counter in distributions.items():
                value = (row.get(field) or '').strip()
                if value:
                    counter[value] += 1

    return {
        'path': path.relative_to(ROOT).as_posix(), 'dataset_kind': kind, 'encoding': encoding,
        'size_bytes': path.stat().st_size, 'fields': fields, 'records': record_count,
        'date_range': [min(dates), max(dates)] if dates else None,
        'invalid_date_count': invalid_date_count, 'unique_stocks': len(stocks),
        'duplicate_key_count': duplicate_key_count,
        'stock_codes': sorted(stocks) if kind == 'daily_trading' else [],
        'missing_by_field': dict(missing),
        'non_numeric_by_field': dict(non_numeric), 'numeric': numeric,
        'distributions': {field: dict(counter) for field, counter in distributions.items() if counter},
    }

files = sorted(path for path in RAW_DIR.rglob('*.csv') if '[DES]' not in path.name)
results = [scan_file(path) for path in files]

# Compare exact keys only for shared codes in the same export period. Distinct export
# periods have non-overlapping observed date ranges, so cannot share a (Stkcd, Trddt) key.
daily = [result for result in results if result['dataset_kind'] == 'daily_trading' and result['date_range']]
daily_periods = defaultdict(list)
for result in daily:
    daily_periods[str(Path(result['path']).parent)].append(result)
candidate_pairs, codes_to_check = [], defaultdict(set)
for period, shards in daily_periods.items():
    for left, right in combinations(shards, 2):
        shared = set(left['stock_codes']) & set(right['stock_codes'])
        if shared:
            candidate_pairs.append((left, right, shared))
            codes_to_check[left['path']].update(shared)
            codes_to_check[right['path']].update(shared)

def keys_for_shared_codes(result, codes):
    keys = set()
    handle, _ = open_csv(ROOT / result['path'])
    with handle:
        for row in csv.DictReader(handle):
            code = (row.get('Stkcd') or '').strip()
            if code in codes:
                keys.add((code, (row.get('Trddt') or '').strip()))
    return keys

candidate_keys = {path: keys_for_shared_codes(next(r for r in daily if r['path'] == path), codes)
                  for path, codes in codes_to_check.items()}
cross_shard_duplicates = []
for left, right, shared in candidate_pairs:
    duplicate_count = len(candidate_keys[left['path']] & candidate_keys[right['path']])
    cross_shard_duplicates.append({
        'left': left['path'], 'right': right['path'], 'shared_stock_count': len(shared),
        'cross_shard_duplicate_key_count': duplicate_count,
    })
for result in daily:
    result.pop('stock_codes', None)

summary = {
    'field_decisions': FIELD_DECISIONS,
    'source_directory': RAW_DIR.relative_to(ROOT).as_posix(),
    'file_count': len(results),
    'files': results,
    'daily_cross_shard_duplicate_check': cross_shard_duplicates,
    'limitations': [
        '本检查不筛选股票池，也不处理 ST、停牌、退市或无效 PE。',
        'H00300 的 Idxtrd08 单位与后续基准收益计算方式留待阶段 7 确认。',
        'ROE 仅检查合并报表、权益、净利润和实际披露日的连接可得性，不在本阶段计算。',
    ],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "field_decisions": {
    "strategy_return_field": "Dretwd",
    "momentum_price_field": "Adjprcwd",
    "pe_field": "PE1TTM",
    "benchmark_code": "H00300"
  },
  "source_directory": "项目初始数据",
  "file_count": 38,
  "files": [
    {
      "path": "项目初始数据/PE(14-16)/STK_MKT_ValuationMetrics.csv",
      "dataset_kind": "valuation",
      "encoding": "utf-8-sig",
      "size_bytes": 56091954,
      "fields": [
        "Symbol",
        "TradingDate",
        "PE1A",
        "PE1B",
        "PE1TTM"
      ],
      "records": 1000000,
      "date_range": [
        "2014-09-12",
        "2016-09-09"
      ],
      "invalid_date_count": 0,
      "unique_stocks": 2027,
      "duplicate_key_count": 0,
      "stock_codes": [],
      "missing_by_field": {
        "PE1B": 153018,
        "PE1TTM": 167074,
        "PE1A": 103842
      },
      "non_numeric_by_field": {},
      "numeric": {
        "PE1TTM": {
          "min": 0.29377,
          "max": 508466.002558,
          "nonpositive": 0
  

In [2]:
# ROE 输入的可用性：仅连接键和实际披露日，不计算 ROE。
def valid_actual_disclosure(value):
    return parse_date((value or '').strip()) is not None

def financial_keys(filename, value_field):
    path = next(path for path in files if path.name == filename)
    keys = set()
    handle, _ = open_csv(path)
    with handle:
        for row in csv.DictReader(handle):
            if (row.get('Typrep') or '').strip() == 'A' and (row.get(value_field) or '').strip():
                keys.add(((row.get('Stkcd') or '').strip(), (row.get('Accper') or '').strip()))
    return keys

income_keys = financial_keys('FS_Comins.csv', 'B002000000')
equity_keys = financial_keys('FS_Combas.csv', 'A003000000')
disclosure_path = next(path for path in files if path.name == 'IAR_Forecdt.csv')
disclosed_keys = set()
handle, _ = open_csv(disclosure_path)
with handle:
    for row in csv.DictReader(handle):
        if valid_actual_disclosure(row.get('Actudt')):
            disclosed_keys.add(((row.get('Stkcd') or '').strip(), (row.get('Accper') or '').strip()))
profit_equity_keys = income_keys & equity_keys
roe_availability = {
    'income_a_nonmissing': len(income_keys),
    'equity_a_nonmissing': len(equity_keys),
    'profit_equity_join': len(profit_equity_keys),
    'join_with_valid_actual_disclosure_date': len(profit_equity_keys & disclosed_keys),
    'join_without_valid_actual_disclosure_date': len(profit_equity_keys - disclosed_keys),
    'income_only': len(income_keys - equity_keys),
    'equity_only': len(equity_keys - income_keys),
}
print(json.dumps(roe_availability, ensure_ascii=False, indent=2))


{
  "income_a_nonmissing": 308004,
  "equity_a_nonmissing": 317735,
  "profit_equity_join": 307993,
  "join_with_valid_actual_disclosure_date": 238531,
  "join_without_valid_actual_disclosure_date": 69462,
  "income_only": 11,
  "equity_only": 9742
}
